# Week 11: Continuous-Time Diffusion and Flow Matching

In Weeks 9 and 10 we worked with **discrete** timesteps $t \in \{0, 1, \ldots, T-1\}$.  
This week we ask: *what happens as $T \to \infty$?*

We will:
1. Reformulate discrete DDPM as a **continuous-time VP-SDE** with $t \in [0, 1]$
2. Show that the same score model can be sampled with **any number of ODE steps** at inference time
3. Introduce **flow matching** (Lipman et al. 2022 / Liu et al. 2022) — a simpler training framework using straight-line transport paths
4. Scale flow matching to MNIST images

**Datasets used:**
- Cat 2D point cloud (from `cat.png`, same as Week 5)
- MNIST digits 0 and 1

`# TODO` marks exercises for students.

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import PIL.Image
from scipy.ndimage import gaussian_filter
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import torchvision

torch.manual_seed(42)
np.random.seed(42)

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

---
## Part 1: From Discrete DDPM to the Continuous-Time VP-SDE

In Week 9, the discrete noise schedule gave us $\bar\alpha_n = \prod_{k=1}^n (1 - \beta_k)$ and the forward process
$$q(x_n \mid x_0) = \mathcal{N}(x_n;\, \sqrt{\bar\alpha_n}\, x_0,\; (1 - \bar\alpha_n) I).$$

As $T \to \infty$ with step size $\Delta t = 1/T$, the discrete schedule converges to the **Variance-Preserving SDE (VP-SDE)**:
$$dx = -\tfrac{1}{2}\beta(t)\, x\, dt + \sqrt{\beta(t)}\, dW, \quad t \in [0,1]$$

with a linear schedule $\beta(t) = \beta_{\min} + (\beta_{\max} - \beta_{\min})\, t$.

The marginal at time $t$ is:
$$q(x_t \mid x_0) = \mathcal{N}(x_t;\; \alpha(t)\, x_0,\; \sigma(t)^2 I)$$

where
$$\alpha(t) = \exp\!\left(-\tfrac{1}{2}\int_0^t \beta(s)\,ds\right), \qquad \sigma(t) = \sqrt{1 - \alpha(t)^2}.$$

**Key insight:** training is unchanged — we still minimise
$$\mathcal{L} = \mathbb{E}_{t \sim \mathcal{U}[0,1],\, x_0,\, \varepsilon}\bigl[\|\varepsilon_\theta(x_t, t) - \varepsilon\|^2\bigr]$$
but now $t$ is a **real number** and we can use *any* ODE solver with *any* number of steps at sampling time.

In [ ]:
# ── Cat 2D point cloud ──────────────────────────────────────────────────────
def get_cat_samples(n_samples=10000, sigma=2.0):
    """Sample 2D points from a blurred cat image via rejection sampling."""
    try:
        cat = PIL.Image.open('cat.png').convert('L')
        arr = 1.0 - (np.array(cat) / 255.0)  # invert: cat outline is high density
        smoothed = gaussian_filter(arr, sigma=sigma)
        prob_map = smoothed / smoothed.sum()
    except FileNotFoundError:
        print('cat.png not found — using Gaussian fallback')
        return torch.randn(n_samples, 2) * 0.5

    flat = prob_map.flatten()
    idx  = np.random.choice(len(flat), size=n_samples, p=flat)
    rows, cols = np.unravel_index(idx, prob_map.shape)

    pts = np.zeros((n_samples, 2))
    pts[:, 0] = (cols / prob_map.shape[1]) * 2 - 1
    pts[:, 1] = 1 - (rows / prob_map.shape[0]) * 2
    pts += np.random.normal(0, 0.01, pts.shape)
    return torch.tensor(pts, dtype=torch.float32)

cat_data = get_cat_samples(10000, sigma=3.0)

plt.figure(figsize=(5, 5))
plt.scatter(cat_data[:, 0], cat_data[:, 1], s=1, alpha=0.3, color='purple')
plt.title('Cat 2D point cloud')
plt.axis('equal'); plt.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# ── Continuous VP-SDE schedule ───────────────────────────────────────────────
beta_min = 0.1
beta_max = 20.0

def alpha(t: torch.Tensor) -> torch.Tensor:
    """Signal coefficient: alpha(0)=1 (clean data), alpha(1)≈0 (pure noise)."""
    return torch.exp(-0.5 * (beta_min * t + 0.5 * (beta_max - beta_min) * t**2))

def sigma(t: torch.Tensor) -> torch.Tensor:
    """Noise coefficient: sigma(0)=0, sigma(1)≈1."""
    return torch.sqrt(torch.clamp(1.0 - alpha(t)**2, min=1e-8))

def q_sample(x_0: torch.Tensor, t: torch.Tensor, noise=None):
    """Forward process: x_t = alpha(t)*x_0 + sigma(t)*eps."""
    if noise is None:
        noise = torch.randn_like(x_0)
    shape = (-1,) + (1,) * (x_0.dim() - 1)  # broadcast over spatial dims
    a = alpha(t).view(*shape)
    s = sigma(t).view(*shape)
    return a * x_0 + s * noise, noise

# ── Visualise alpha(t) and sigma(t) ─────────────────────────────────────────
t_vals = torch.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t_vals, alpha(t_vals).numpy(), label=r'$\alpha(t)$')
ax.plot(t_vals, sigma(t_vals).numpy(), label=r'$\sigma(t)$')
ax.set_xlabel('t'); ax.legend(); ax.set_title('VP-SDE continuous schedule')
plt.tight_layout(); plt.show()

# ── Show cat data corrupted at different t ───────────────────────────────────
t_show = [0.0, 0.25, 0.5, 0.75, 1.0]
n_show = 3000
x0_show = cat_data[:n_show].to(device)

fig, axes = plt.subplots(1, len(t_show), figsize=(15, 3))
for ax, tv in zip(axes, t_show):
    t_batch = torch.full((n_show,), tv, device=device)
    xt, _ = q_sample(x0_show, t_batch)
    xt = xt.cpu().numpy()
    ax.scatter(xt[:, 0], xt[:, 1], s=1, alpha=0.3)
    ax.set_title(f't = {tv}')
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.axis('off')
plt.suptitle('Cat data corrupted by VP-SDE forward process', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── Verify: discrete alpha_cumprod ≈ continuous alpha(t) ────────────────────
T = 1000
betas_disc = torch.linspace(beta_min / T, beta_max / T, T)
alphas_cumprod_disc = torch.cumprod(1.0 - betas_disc, dim=0).sqrt()

t_grid = torch.linspace(1/T, 1.0, T)
alphas_cont = alpha(t_grid)

plt.figure(figsize=(7, 3))
plt.plot(t_grid.numpy(), alphas_cumprod_disc.numpy(), label=r'$\sqrt{\bar\alpha_n}$ (discrete, T=1000)', lw=2)
plt.plot(t_grid.numpy(), alphas_cont.numpy(), '--', label=r'$\alpha(t)$ (continuous)', lw=2)
plt.xlabel('t = n/T'); plt.legend()
plt.title('Discrete schedule converges to continuous $\\alpha(t)$')
plt.tight_layout(); plt.show()

---
## Part 2: Continuous-Time Score Model and ODE Sampling

### The Probability Flow ODE

The VP-SDE has a deterministic counterpart: the **probability flow ODE**
$$\frac{dx}{dt} = -\tfrac{1}{2}\beta(t)\bigl[x + \nabla_x \log p_t(x)\bigr]$$
whose marginals match the SDE at every $t$. We can generate samples by integrating this ODE from $t=1$ (noise) to $t=0$ (data) with **no stochasticity**.

Given $\varepsilon$-prediction, the score is $\nabla_x \log p_t(x_t) \approx -\varepsilon_\theta(x_t, t) / \sigma(t)$.  
Substituting and simplifying yields the **DDIM-style update** for step $\Delta t$:
$$x_{t - \Delta t} = \frac{\alpha(t-\Delta t)}{\alpha(t)}\, x_t \;+\; \left(\sigma(t-\Delta t) - \frac{\alpha(t-\Delta t)}{\alpha(t)}\, \sigma(t)\right)\varepsilon_\theta(x_t, t)$$

This is the **same model as Week 9/10**, but now we can choose any $N$ at sampling time.

In [ ]:
class ContinuousMLP(nn.Module):
    """
    MLP for continuous-time epsilon-prediction.
    t ∈ [0,1] is embedded via [sin(pi*t), cos(pi*t)] -> hidden_dim.
    """
    def __init__(self, hidden_dim=256):
        super().__init__()
        self.time_embed = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.GELU(),
        )
        self.net = nn.Sequential(
            nn.Linear(2 + hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),      nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),      nn.GELU(),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        # t: (B,) float in [0, 1]
        t_emb = self.time_embed(
            torch.stack([torch.sin(t * math.pi), torch.cos(t * math.pi)], dim=-1)
        )
        return self.net(torch.cat([x, t_emb], dim=-1))

In [ ]:
# ── Training: continuous-time epsilon prediction ─────────────────────────────
cat_data = get_cat_samples(10000, sigma=3.0)
dataloader = DataLoader(TensorDataset(cat_data), batch_size=256, shuffle=True)

score_model = ContinuousMLP().to(device)
opt = torch.optim.Adam(score_model.parameters(), lr=1e-3)

losses = []
for epoch in range(300):
    for (x_0,) in dataloader:
        x_0 = x_0.to(device)
        B = x_0.shape[0]

        # Sample t ~ Uniform[0, 1]  (continuous, not discrete integers)
        t = torch.rand(B, device=device)

        # Corrupt data
        x_t, noise = q_sample(x_0, t)

        eps_pred = score_model(x_t, t)
        loss = F.mse_loss(eps_pred, noise)

        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())

    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1:3d} | loss {np.mean(losses[-len(dataloader):]):.4f}')

plt.figure(figsize=(7, 3))
plt.plot(losses[::10]); plt.xlabel('steps (×10)'); plt.ylabel('MSE loss')
plt.title('Continuous VP-SDE score model training')
plt.tight_layout(); plt.show()

In [ ]:
@torch.no_grad()
def sample_ode(model, n_samples=2000, N=100):
    """
    Euler solver for the probability flow ODE.
    Integrates from t=1 (noise) to t≈0 (data) in N steps.

    DDIM-style update:
        x_{t-dt} = (alpha(t-dt)/alpha(t)) * x_t
                 + (sigma(t-dt) - (alpha(t-dt)/alpha(t)) * sigma(t)) * eps_pred
    """
    model.eval()
    x = torch.randn(n_samples, 2, device=device)
    ts = torch.linspace(1.0, 1e-3, N + 1, device=device)

    for i in range(N):
        t_curr = ts[i]
        t_next = ts[i + 1]

        eps_pred = model(x, t_curr.expand(n_samples))

        a_c = alpha(t_curr)
        a_n = alpha(t_next)
        s_c = sigma(t_curr)
        s_n = sigma(t_next)

        # DDIM update step
        x = (a_n / a_c) * x + (s_n - (a_n / a_c) * s_c) * eps_pred

    return x.cpu().numpy()

In [ ]:
# ── Explore: sample quality vs number of ODE steps ──────────────────────────
step_counts = [5, 10, 20, 50, 100, 500]
fig, axes = plt.subplots(1, len(step_counts), figsize=(18, 3))

for ax, N in zip(axes, step_counts):
    samples = sample_ode(score_model, n_samples=3000, N=N)
    ax.scatter(samples[:, 0], samples[:, 1], s=1, alpha=0.4, color='steelblue')
    ax.set_title(f'N = {N}')
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.axis('off')

plt.suptitle('VP-SDE Probability Flow ODE: sample quality vs steps', y=1.02)
plt.tight_layout(); plt.show()

### Discussion

1. At what value of $N$ does the cat shape first become recognisable? At what $N$ does it look sharp?
2. The ODE sampler is **deterministic** — for a fixed initial noise, it always produces the same output. What does this imply about diversity compared to stochastic DDPM?
3. Why might VP-SDE ODE paths be curved (requiring more steps) rather than straight?

---
## Part 3: Flow Matching on Cat Data

**Flow matching** (Lipman et al. 2022; Liu et al. 2022 *Rectified Flow*) is a fundamentally simpler training framework.  
Instead of learning a score and relying on the SDE/ODE connection, we directly learn a **vector field** that transports noise to data along **straight-line paths**.

**Convention** (opposite to VP-SDE above!):

| $t$ | meaning |
|---|---|
| $0$ | noise $\varepsilon \sim \mathcal{N}(0, I)$ |
| $1$ | data $x_0 \sim p_{\text{data}}$ |

**Forward path** (training only, no SDE):
$$x_t = (1-t)\,\varepsilon + t\,x_0, \qquad \varepsilon \sim \mathcal{N}(0,I),\; x_0 \sim p_{\text{data}}$$

The **conditional vector field** along this path is constant:
$$u_t(x_t \mid x_0, \varepsilon) = x_0 - \varepsilon$$

**Training loss** (conditional flow matching, CFM):
$$\mathcal{L}_{\text{CFM}} = \mathbb{E}_{t \sim \mathcal{U}[0,1],\, x_0 \sim p_{\text{data}},\, \varepsilon \sim \mathcal{N}(0,I)} \bigl[\|v_\theta(x_t, t) - (x_0 - \varepsilon)\|^2\bigr]$$

**Sampling** — Euler from $t=0$ to $t=1$:
$$x_{t + \Delta t} = x_t + \Delta t \cdot v_\theta(x_t, t)$$

No noise schedule, no $\alpha(t)$, no $\beta(t)$.

In [ ]:
def flow_interpolate(x_0: torch.Tensor, eps: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """
    Linear interpolation: x_t = (1-t)*eps + t*x_0
    x_0, eps: (B, D)  |  t: (B,)
    """
    t = t.view(-1, *([1] * (x_0.dim() - 1)))  # broadcast
    return (1 - t) * eps + t * x_0


def flow_target(x_0: torch.Tensor, eps: torch.Tensor) -> torch.Tensor:
    """Target vector field: constant direction from noise to data."""
    return x_0 - eps

In [ ]:
class FlowMLP(nn.Module):
    """
    Same architecture as ContinuousMLP — only the training objective and
    sampling direction differ.
    Model now predicts velocity v_theta(x_t, t) instead of noise epsilon.
    """
    def __init__(self, hidden_dim=256):
        super().__init__()
        self.time_embed = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.GELU(),
        )
        self.net = nn.Sequential(
            nn.Linear(2 + hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),      nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),      nn.GELU(),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = self.time_embed(
            torch.stack([torch.sin(t * math.pi), torch.cos(t * math.pi)], dim=-1)
        )
        return self.net(torch.cat([x, t_emb], dim=-1))

In [ ]:
# ── Flow matching training ───────────────────────────────────────────────────
flow_model = FlowMLP().to(device)
opt_flow = torch.optim.Adam(flow_model.parameters(), lr=1e-3)

losses_flow = []
for epoch in range(300):
    for (x_0,) in dataloader:
        x_0 = x_0.to(device)
        B = x_0.shape[0]

        t   = torch.rand(B, device=device)
        eps = torch.randn_like(x_0)

        x_t    = flow_interpolate(x_0, eps, t)
        target = flow_target(x_0, eps)

        v_pred = flow_model(x_t, t)
        loss = F.mse_loss(v_pred, target)

        opt_flow.zero_grad(); loss.backward(); opt_flow.step()
        losses_flow.append(loss.item())

    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1:3d} | loss {np.mean(losses_flow[-len(dataloader):]):.4f}')

plt.figure(figsize=(7, 3))
plt.plot(losses_flow[::10]); plt.xlabel('steps (×10)'); plt.ylabel('MSE loss')
plt.title('Flow matching training')
plt.tight_layout(); plt.show()

In [ ]:
@torch.no_grad()
def sample_flow(model, n_samples=2000, N=100):
    """
    Euler integration from t=0 (noise) to t=1 (data).
    x_{t+dt} = x_t + dt * v_theta(x_t, t)

    Note: direction is OPPOSITE to the VP-SDE ODE (noise->data, not data->noise).
    """
    model.eval()
    x = torch.randn(n_samples, 2, device=device)
    dt = 1.0 / N

    for i in range(N):
        t = torch.full((n_samples,), i * dt, device=device)
        v = model(x, t)
        x = x + dt * v

    return x.cpu().numpy()

In [ ]:
# ── Side-by-side: VP-SDE ODE vs Flow Matching at different N ────────────────
step_counts = [5, 10, 20, 50, 100, 500]
fig, axes = plt.subplots(2, len(step_counts), figsize=(18, 6))

for col, N in enumerate(step_counts):
    s1 = sample_ode(score_model, n_samples=3000, N=N)
    axes[0, col].scatter(s1[:, 0], s1[:, 1], s=1, alpha=0.4, color='steelblue')
    axes[0, col].set_title(f'N = {N}')

    s2 = sample_flow(flow_model, n_samples=3000, N=N)
    axes[1, col].scatter(s2[:, 0], s2[:, 1], s=1, alpha=0.4, color='darkorange')

    for ax in axes[:, col]:
        ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.axis('off')

axes[0, 0].set_ylabel('VP-SDE ODE', fontsize=11)
axes[1, 0].set_ylabel('Flow Matching', fontsize=11)

plt.suptitle('VP-SDE ODE (top) vs Flow Matching (bottom): steps vs quality', y=1.02)
plt.tight_layout(); plt.show()

### Discussion

1. Flow matching uses **straight-line paths**. Why does this allow good generation with fewer Euler steps than the curved VP-SDE ODE?
2. Both models have the same architecture and are trained with the same MSE loss. What is fundamentally different about what each model outputs?
3. Flow matching has **no noise schedule**. Does this make it strictly simpler than score-based diffusion, or are there trade-offs?

---
## Part 4: Flow Matching on MNIST (Digits 0 and 1)

We now scale up from 2D point clouds to **28×28 grayscale images**, using a small U-Net (`MiniUNetFlow` from `unet_flow.py`) as the velocity network.

The training objective and Euler sampler are identical to Part 3 — only the model architecture and data dimensions change.

In [ ]:
# ── MNIST digits 0 and 1 ─────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),   # map to [-1, 1]
])
mnist_train = datasets.MNIST('./data', train=True, download=True, transform=transform)

# Filter to digits 0 and 1
idx = (mnist_train.targets == 0) | (mnist_train.targets == 1)
mnist_train.targets = mnist_train.targets[idx]
mnist_train.data    = mnist_train.data[idx]

img_loader = DataLoader(mnist_train, batch_size=128, shuffle=True)
print(f'MNIST 0&1 training samples: {len(mnist_train)}')

# Quick preview
x_preview, _ = next(iter(img_loader))
grid = torchvision.utils.make_grid(x_preview[:16], nrow=8, normalize=True)
plt.figure(figsize=(10, 2))
plt.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
plt.axis('off'); plt.title('MNIST digits 0 and 1')
plt.tight_layout(); plt.show()

In [ ]:
from unet_flow import MiniUNetFlow

flow_unet = MiniUNetFlow(in_channels=1, time_emb_dim=64).to(device)
print(f'MiniUNetFlow parameters: {sum(p.numel() for p in flow_unet.parameters()):,}')

In [ ]:
# ── Flow matching training on MNIST images ───────────────────────────────────
opt_img = torch.optim.Adam(flow_unet.parameters(), lr=1e-3)
epochs_img = 20
losses_img = []

for epoch in range(epochs_img):
    for x_0, _ in img_loader:
        x_0 = x_0.to(device)
        B = x_0.shape[0]

        t   = torch.rand(B, device=device)
        eps = torch.randn_like(x_0)

        # flow_interpolate handles 4D tensors via the broadcast in view()
        x_t    = flow_interpolate(x_0, eps, t)
        target = flow_target(x_0, eps)        # x_0 - eps

        v_pred = flow_unet(x_t, t)
        loss   = F.mse_loss(v_pred, target)

        opt_img.zero_grad(); loss.backward(); opt_img.step()
        losses_img.append(loss.item())

    print(f'Epoch {epoch+1:2d}/{epochs_img} | loss {np.mean(losses_img[-len(img_loader):]):.4f}')

plt.figure(figsize=(7, 3))
plt.plot(losses_img); plt.xlabel('batch'); plt.ylabel('MSE loss')
plt.title('Flow matching training on MNIST')
plt.tight_layout(); plt.show()

In [ ]:
@torch.no_grad()
def sample_flow_mnist(model, n_samples=8, N=100):
    """
    Euler integration from t=0 (noise) to t=1 (data) for images.
    x_{t+dt} = x_t + dt * v_theta(x_t, t)
    """
    model.eval()
    x = torch.randn(n_samples, 1, 28, 28, device=device)
    dt = 1.0 / N

    for i in range(N):
        t = torch.full((n_samples,), i * dt, device=device)
        v = model(x, t)
        x = x + dt * v

    return x.cpu()

In [ ]:
# ── Explore: image quality vs number of Euler steps ─────────────────────────
step_counts = [5, 10, 20, 50, 100, 500]
n_per_row   = 8

fig, axes = plt.subplots(len(step_counts), n_per_row, figsize=(16, 2.5 * len(step_counts)))

for row, N in enumerate(step_counts):
    imgs = sample_flow_mnist(flow_unet, n_samples=n_per_row, N=N)
    for col in range(n_per_row):
        img = (imgs[col].squeeze().numpy() + 1) / 2   # [-1,1] -> [0,1]
        img = np.clip(img, 0, 1)
        axes[row, col].imshow(img, cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'N = {N}', fontsize=11)

plt.suptitle('Flow Matching on MNIST: sample quality vs Euler steps', y=1.01)
plt.tight_layout(); plt.show()

### Final Discussion

1. Compare the minimum viable $N$ for the cat 2D experiment (Part 3) vs MNIST (Part 4). Why do images require more steps for recognisable quality?
2. In Parts 3 and 4 we trained a single unconditional model on digits 0 *and* 1 mixed together. What would you expect to see in the samples — pure 0s, pure 1s, or a mix? Does your observation match the expectation?
3. **Connecting Weeks 9–11:**  
   - Week 9: discrete DDPM, score matching  
   - Week 10: conditioning via classifier guidance and CFG  
   - Week 11: continuous time, ODE sampling, flow matching  
   
   What is the single most important idea connecting all three frameworks?